# 09 — Ctf-DE / Ctf-IE / Ctf-SE Estimation via `faircause`

Continuation of notebooks 01–06. Runs Plečko & Bareinboim's causal fairness decomposition (their own R package, `faircause`) on the project's density/reliability/topology/disparity structure. Currently uses synthetic data matching the DAG — swap in your real METR-LA-derived signals (from notebooks 02, 05, 06) once available.

**Important, confirmed from the authors' own documentation:** `faircause` expects a binary/categorical treatment (`x0`/`x1` reference levels). Sensor density is continuous, so it's discretized via median split below — a real design choice, flagged explicitly.


In [1]:
# --- R environment setup for this notebook (safe to re-run; skips if already installed) ---
import subprocess
import sys
subprocess.run(["apt-get", "install", "-y", "-qq", "r-base-core"], stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rpy2"])
get_ipython().run_line_magic("load_ext", "rpy2.ipython")
print("R + rpy2 bridge ready.")


R + rpy2 bridge ready.


In [2]:
%%R
if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")
if (!requireNamespace("faircause", quietly = TRUE)) devtools::install_github("dplecko/CFA")
library(faircause)
cat("faircause loaded, version:", as.character(packageVersion("faircause")), "\n")


── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/RtmpCUSOKT/remotes1e4a9a255a8/dplecko-CFA-1d0dc97/DESCRIPTION’ ... OK
* preparing ‘faircause’:
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
Removed empty directory ‘faircause/vignettes’
* building ‘faircause_0.4.0.tar.gz’

Installing 10 packages: RcppEigen, sandwich, lmtest, RcppTOML, xgboost, grf, reticulate, latex2exp, assertthat, ranger
Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/RcppEigen_0.3.4.0.2.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/sandwich_3.1-2.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/lmtest_0.9-40.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RcppTOML_0.2.3.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/xgboos

RInterpreterError: Failed to parse and evaluate line 'if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")\nif (!requireNamespace("faircause", quietly = TRUE)) devtools::install_github("dplecko/CFA")\nlibrary(faircause)\ncat("faircause loaded, version:", as.character(packageVersion("faircause")), "\\n")\n'.
R error message: 'Error in library(faircause) : there is no package called ‘faircause’'
R stdout:
Downloading GitHub repo dplecko/CFA@HEAD
Installing 10 packages: RcppEigen, sandwich, lmtest, RcppTOML, xgboost, grf, reticulate, latex2exp, assertthat, ranger
Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/RcppEigen_0.3.4.0.2.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/sandwich_3.1-2.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/lmtest_0.9-40.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RcppTOML_0.2.3.tar.gz'
trying URL 'https://cran.rstudi

In [ ]:
%%R
#!/usr/bin/env Rscript
# ============================================================================
# 02_ctf_estimation_faircause.R
#
# Run in Colab via %%R after 00_setup_and_fairtp_verified.ipynb has installed
# faircause. NOT executed/tested locally (no CRAN access in the build
# environment) — written against faircause's documented SFM projection
# pattern, verified from the authors' own vignettes (dplecko.github.io/CFA).
# ============================================================================

library(faircause)

1. Load your consolidated project data (metr_la_metrics.csv)

In [ ]:
%%R
synthetic_data <- read.csv("metr_la_metrics.csv")
synthetic_data$reliability <- 1.0 - (0.6 * synthetic_data$zero_rate + 0.2 * synthetic_data$cusum_flag_rate + 0.2 * synthetic_data$ewma_flag_rate)
synthetic_data$disparity <- synthetic_data$persistence_error


2. Discretize density into a binary treatment (median split — replace with
   a more principled cutoff, e.g. a policy-relevant threshold, once real
   data is available and you have a reason to prefer one over the median)

In [ ]:
%%R
median_density <- median(synthetic_data$density)
synthetic_data$density_bin <- ifelse(synthetic_data$density > median_density,
                                       "high_density", "low_density")

cat("Density discretized at median (", round(median_density, 2), ").\n")
cat("Group sizes:\n")
print(table(synthetic_data$density_bin))
cat("\nCHECK POSITIVITY: neither group should be near-empty. If one group has\n")
cat("very few sensors, the causal estimate will be unstable regardless of the\n")
cat("estimation method used.\n\n")

3. Specify the SFM (Standard Fairness Model) projection
   X = treatment, Z = confounders, W = mediators, Y = outcome

In [ ]:
%%R
X <- "density_bin"
Z <- c("traffic_regime", "road_type")
W <- c("reliability", "topology")
Y <- "disparity"

4. Run the Ctf-DE/Ctf-IE/Ctf-SE decomposition

In [ ]:
%%R
result <- fairness_cookbook(
  data = synthetic_data,
  X = X, Z = Z, W = W, Y = Y,
  x0 = "low_density", x1 = "high_density"
)
# NOTE: the exact function name may be `fairness_cookbook()` or `faircause()`
# depending on the package version at the time you run this — v0.2.0 is
# under active development. If `fairness_cookbook` errors with "could not
# find function", check `ls("package:faircause")` to see the current name
# and adjust the call above accordingly.

cat("=== Ctf-DE / Ctf-IE / Ctf-SE decomposition ===\n")
print(summary(result))

5. Effect Estimation Sanity Check

In [ ]:
%%R
cat("\n=== Effect Size Interpretation ===\n")
cat("Examine the decomposition above. If the Ctf-IE for reliability is\n")
cat("substantial compared to topology and direct effects, the primary\n")
cat("source of forecast disparity stems from sensor degradation.\n")


6. Export for use in downstream comparison against FairTP's RSF/SDF

In [ ]:
%%R
write.csv(as.data.frame(summary(result)), "ctf_decomposition_results.csv", row.names = FALSE)
cat("\nResults saved to ctf_decomposition_results.csv\n")